CELL 1: Imports and Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Imports complete")
print(f"Pandas version: {pd.__version__}")

✅ Imports complete
Pandas version: 3.0.2


CELL 2: Load Full Dataset (with optimized dtypes)

In [5]:
data_path = Path("../data/raw/twcs.csv")

# Define dtypes to save memory
dtypes = {
    'tweet_id': 'int32',
    'author_id': 'object',
    'inbound': 'bool',
    'created_at': 'object',
    'text': 'object',
    'response_tweet_id': 'object',
    'in_response_to_tweet_id': 'object'
}

print(f"Loading from: {data_path}")
print(f"File size: {data_path.stat().st_size / (1024**3):.2f} GB")
print("This may take 1-2 minutes...")

# Load FULL dataset
df = pd.read_csv(data_path, dtype=dtypes, low_memory=False)

print(f"\n✅ Loaded {len(df):,} rows")
print(f"Shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

Loading from: ..\data\raw\twcs.csv
File size: 0.48 GB
This may take 1-2 minutes...

✅ Loaded 2,811,774 rows
Shape: (2811774, 7)
Memory usage: 1273.1 MB


CELL 3: Basic Information

In [6]:
print("=" * 50)
print("BASIC INFORMATION")
print("=" * 50)

print(f"\nRows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print(f"\nColumn names and types:")
for col in df.columns:
    print(f"  {col:<25} {df[col].dtype}")

print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

BASIC INFORMATION

Rows: 2,811,774
Columns: 7

Column names and types:
  tweet_id                  int32
  author_id                 object
  inbound                   bool
  created_at                object
  text                      object
  response_tweet_id         object
  in_response_to_tweet_id   object

Memory usage: 1273.1 MB


Question: response_tweet_id has values like "5,7" - we put it as object or string?

Answer: object (string) is correct.

Why? Because pandas reads comma-separated values as strings. We keep it as object now, then in Phase 3 we will:

    Split the string by comma: "5,7" → ['5', '7']

    Convert each to integer: [5, 7]

    Expand into separate rows or create a mapping

This is correct.

CELL 4: First 5 Rows (Preview)

In [7]:
print("=" * 50)
print("FIRST 5 ROWS")
print("=" * 50)
print(df.head().to_string())

FIRST 5 ROWS
   tweet_id   author_id  inbound                      created_at                                                                                                                          text response_tweet_id in_response_to_tweet_id
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017     @115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.                 2                       3
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017                                                                                 @sprintcare and how do you propose we do that               NaN                       1
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017                                            @sprintcare I have sent several private messages and no one is responding as usual                 1                       4
3         4  sprintcare    False  Tue Oct 31 21:54:49 +

CELL 5: Missing Values

In [8]:
print("=" * 50)
print("MISSING VALUES")
print("=" * 50)

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing': missing,
    'Percent': missing_pct
}).sort_values('Missing', ascending=False)

missing_df = missing_df[missing_df['Missing'] > 0]

if len(missing_df) > 0:
    print(missing_df.to_string())
else:
    print("No missing values found")

MISSING VALUES
                         Missing    Percent
response_tweet_id        1040629  37.009696
in_response_to_tweet_id   794335  28.250315


Are we sure of this?

Yes, this is correct. Here's what it means:
Column	Missing	Meaning
response_tweet_id	37% missing	37% of tweets are conversation starters (no parent)
in_response_to_tweet_id	28% missing	28% have no alternative parent reference

This makes sense because:

    Not every tweet is a response to another tweet

    Some tweets are original customer questions (no parent)

    The two columns have different coverage (63% have response_tweet_id, 72% have in_response_to_tweet_id)

Conclusion: This is expected and correct.

CELL 6: Inbound/Outbound Distribution

In [9]:
print("=" * 50)
print("INBOUND VS OUTBOUND")
print("=" * 50)

if 'inbound' in df.columns:
    inbound_count = df['inbound'].sum()
    outbound_count = len(df) - inbound_count
    
    print(f"Inbound (customer question):  {inbound_count:,} ({inbound_count/len(df)*100:.1f}%)")
    print(f"Outbound (brand response):    {outbound_count:,} ({outbound_count/len(df)*100:.1f}%)")
else:
    print("Column 'inbound' not found")

INBOUND VS OUTBOUND
Inbound (customer question):  1,537,843 (54.7%)
Outbound (brand response):    1,273,931 (45.3%)


CELL 7: Response Tweet ID Analysis (Critical)

In [10]:
print("=" * 50)
print("RESPONSE_TWEET_ID ANALYSIS")
print("=" * 50)

if 'response_tweet_id' in df.columns:
    # Count non-null
    has_response = df['response_tweet_id'].notna().sum()
    print(f"Tweets that respond to something: {has_response:,} ({has_response/len(df)*100:.1f}%)")
    
    # Check for comma-separated values (multiple parents)
    response_str = df['response_tweet_id'].dropna().astype(str)
    has_comma = response_str.str.contains(',').sum()
    print(f"\nTweets with MULTIPLE parents (comma-separated): {has_comma:,}")
    
    if has_comma > 0:
        print(f"  → {has_comma/len(df)*100:.2f}% of all tweets have multiple parents")
        print(f"  → {has_comma/has_response*100:.2f}% of responses have multiple parents")
        
        # Show examples
        print("\nExamples of multiple parents:")
        examples = df[df['response_tweet_id'].astype(str).str.contains(',', na=False)].head(3)
        for idx, row in examples.iterrows():
            print(f"\n  tweet_id: {row['tweet_id']}")
            print(f"  response_tweet_id: {row['response_tweet_id']}")
            print(f"  text preview: {str(row['text'])[:80]}...")
    else:
        print("No comma-separated values found")
else:
    print("Column 'response_tweet_id' not found")

RESPONSE_TWEET_ID ANALYSIS
Tweets that respond to something: 1,771,145 (63.0%)

Tweets with MULTIPLE parents (comma-separated): 222,426
  → 7.91% of all tweets have multiple parents
  → 12.56% of responses have multiple parents

Examples of multiple parents:

  tweet_id: 6
  response_tweet_id: 5,7
  text preview: @115712 Can you please send us a private message, so that I can gain further det...

  tweet_id: 8
  response_tweet_id: 9,6,10
  text preview: @sprintcare is the worst customer service...

  tweet_id: 12
  response_tweet_id: 11,13,14
  text preview: @sprintcare You gonna magically change your connectivity for me and my whole fam...


What this tells us:
Finding	Implication
63% of tweets are responses	More replies than original tweets
12.56% of responses have multiple parents	Significant number - we cannot ignore this
222,426 tweets with multiple parents	~222k tweets need special handling
Examples from your output:
text

tweet_id: 6, response_tweet_id: "5,7"  → responds to tweet 5 AND tweet 7
tweet_id: 8, response_tweet_id: "9,6,10" → responds to 3 different tweets

This confirms: We MUST handle multiple parents in Phase 3.

CELL 8: In Response To Tweet ID Analysis

In [11]:
print("=" * 50)
print("IN_RESPONSE_TO_TWEET_ID ANALYSIS")
print("=" * 50)

if 'in_response_to_tweet_id' in df.columns:
    has_value = df['in_response_to_tweet_id'].notna().sum()
    print(f"Rows with value: {has_value:,} ({has_value/len(df)*100:.1f}%)")
    
    # Check for commas
    val_str = df['in_response_to_tweet_id'].dropna().astype(str)
    has_comma = val_str.str.contains(',').sum()
    print(f"Rows with multiple values (comma-separated): {has_comma:,}")
    
    # Show sample
    print("\nSample values (first 5 non-null):")
    sample = df['in_response_to_tweet_id'].dropna().head(5).tolist()
    for val in sample:
        print(f"  {val}")

IN_RESPONSE_TO_TWEET_ID ANALYSIS
Rows with value: 2,017,439 (71.7%)
Rows with multiple values (comma-separated): 0

Sample values (first 5 non-null):
  3
  1
  4
  5
  6


What this tells us:
Finding	Meaning
71.7% have a value	Higher coverage than response_tweet_id (63%)
No comma-separated values	This column is SINGLE value only
Sample values are single numbers	3, 1, 4, 5, 6 - all single IDs
Comparison between the two columns:
Column	Has Value	Has Multiple	Use For
response_tweet_id	63%	✅ YES (12.56% of responses)	Primary parent reference
in_response_to_tweet_id	72%	❌ NO	Alternative/single parent reference
Important Insight:

The two columns serve different purposes:

    response_tweet_id = CAN point to multiple parents (full relationship graph)

    in_response_to_tweet_id = Points to SINGLE parent (simpler reference)

For pairing questions with answers, we should prioritize response_tweet_id because it captures all relationships, even if multiple.

CELL 9: Unique Values Count (for understanding scale)

In [12]:
print("=" * 50)
print("UNIQUE VALUES COUNT")
print("=" * 50)

print(f"Unique tweet_id: {df['tweet_id'].nunique():,}")
print(f"Unique author_id: {df['author_id'].nunique():,}")
print(f"Unique response_tweet_id values: {df['response_tweet_id'].nunique():,}")
print(f"Unique in_response_to_tweet_id values: {df['in_response_to_tweet_id'].nunique():,}")

UNIQUE VALUES COUNT
Unique tweet_id: 2,811,774
Unique author_id: 702,777
Unique response_tweet_id values: 1,771,145
Unique in_response_to_tweet_id values: 1,774,822


CELL 10: Summary of Findings

In [13]:
print("=" * 50)
print("SUMMARY OF FINDINGS")
print("=" * 50)

print(f"""
📊 Dataset Size: {df.shape[0]:,} rows, {df.shape[1]} columns
📊 File Size: {data_path.stat().st_size / (1024**3):.2f} GB

📋 Column Types:
   - tweet_id: int32 (unique identifier)
   - author_id: object (who wrote it)
   - inbound: bool (True = customer, False = brand)
   - created_at: object (timestamp)
   - text: object (content)
   - response_tweet_id: object (parent IDs, CAN be comma-separated)
   - in_response_to_tweet_id: object (alternative parent reference)

⚠️ Critical Findings:
   - response_tweet_id {'DOES' if has_comma > 0 else 'DOES NOT'} contain comma-separated values
   - If YES: Need to parse and expand for conversation mapping

📊 Balance:
   - Inbound (customer): {df['inbound'].sum():,} ({df['inbound'].sum()/len(df)*100:.1f}%)
   - Outbound (brand): {(~df['inbound']).sum():,} ({(~df['inbound']).sum()/len(df)*100:.1f}%)

📊 Missing Values:
   - Columns with missing: {len(missing_df)}
   - Total missing cells: {df.isnull().sum().sum():,}

➡️ Next Phase: Parse response_tweet_id to handle multiple parents
""")

SUMMARY OF FINDINGS

📊 Dataset Size: 2,811,774 rows, 7 columns
📊 File Size: 0.48 GB

📋 Column Types:
   - tweet_id: int32 (unique identifier)
   - author_id: object (who wrote it)
   - inbound: bool (True = customer, False = brand)
   - created_at: object (timestamp)
   - text: object (content)
   - response_tweet_id: object (parent IDs, CAN be comma-separated)
   - in_response_to_tweet_id: object (alternative parent reference)

⚠️ Critical Findings:
   - response_tweet_id DOES NOT contain comma-separated values
   - If YES: Need to parse and expand for conversation mapping

📊 Balance:
   - Inbound (customer): 1,537,843 (54.7%)
   - Outbound (brand): 1,273,931 (45.3%)

📊 Missing Values:
   - Columns with missing: 2
   - Total missing cells: 1,834,964

➡️ Next Phase: Parse response_tweet_id to handle multiple parents



CELL 11: Save Summary (Optional)

In [14]:
# Save summary to log file
summary_path = Path("../logs/phase2_inspection_summary.txt")
summary_path.parent.mkdir(exist_ok=True)

with open(summary_path, 'w') as f:
    f.write("PHASE 2 INSPECTION SUMMARY\n")
    f.write("=" * 40 + "\n\n")
    f.write(f"Total rows: {df.shape[0]:,}\n")
    f.write(f"Total columns: {df.shape[1]}\n")
    f.write(f"File size: {data_path.stat().st_size / (1024**3):.2f} GB\n\n")
    f.write(f"Inbound (customer): {df['inbound'].sum():,}\n")
    f.write(f"Outbound (brand): {(~df['inbound']).sum():,}\n\n")
    f.write(f"Has comma-separated response_tweet_id: {has_comma > 0}\n")
    if has_comma > 0:
        f.write(f"  Count: {has_comma:,}\n")

print(f"✅ Summary saved to: {summary_path}")

✅ Summary saved to: ..\logs\phase2_inspection_summary.txt
